In [1]:
import requests
import pandas as pd
from datetime import date

# Configuración
OPERADORA  = "WOM"
FECHA_HOY  = date.today().isoformat()

# NOTA: WOM requiere SKUs explícitos en la URL. Actualizar si se agregan nuevos modelos.
SKUS = (
    "22101316G,A3517-512GB,Z2474,SM-A566E,SM-S731B,SM-A176BZKOLEL,SM-F766B,"
    "SM-F761B,24129PN74G,XT2307-1,2406APNFAG,XT2429-1,A3409,24115RA8EG,A3517,"
    "NLA-LX3-128GB,24094RAD4G,15C256GB,A3526-512GB,ABR-NX3,Z2464N,25028RN03L,"
    "A3287,XT2535-2-128GB,2506BPN68G,XT2529-1,BRP-NX3,2510ERA8BG,25078RA3EL128GB,"
    "NLA-LX3,KJ8s,X6730B,KL8,A3526,ALT-NX3,XT2535-2,SM-A366E,A3523-512GB,"
    "25069PTEBG,XT2507-1,XT2527-6,LGN-NX3,A3523,X6857,XT2503-1,25057RN09G,"
    "A3090-128GB,25098RA98G,XT2433-2-256GB,A3520,MTN-NX3,SM-A266M,25080RABDG,A3634"
)

URL = f"https://store-srv.wom.cl/rest/V1/content/getGraphqlDataFromSkus?skus={SKUS}"

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://store.wom.cl/equipos/"
}

In [2]:
# Función auxiliar para extraer precios
def extraer_precio(lista_precios, tipo_buscado='price'):
    """Busca un tipo de precio específico en la lista de precios de WOM."""
    for item in lista_precios:
        if item.get('priceType') == tipo_buscado:
            return item.get('price', {}).get('value')
    return None


# Extracción
print(f"Conectando con la API de {OPERADORA}...")
response = requests.get(URL, headers=HEADERS)

if response.status_code != 200:
    raise ConnectionError(f"Error {response.status_code} al conectar con {OPERADORA}")

data = response.json()
lista_wom = []

for producto in data:
    for hijo in producto.get('childs', []):
        g_data      = hijo.get('graphql_data', {}).get('graphql_data', {})
        specs       = g_data.get('productSpecification', {}).get('productSpecCharacteristic', [])
        prices_node = g_data.get('productOfferingPrice', {})

        # Extraer marca desde specs
        marca = "No definida"
        for spec in specs:
            if spec.get('name') == 'marca':
                marca = spec.get('productSpecCharacteristicValue', [{}])[0].get('value', 'No definida')
                break

        # Extraer los 4 tipos de precio
        p_normal       = extraer_precio(prices_node.get('standard',       {}).get('relatedPrice', []))
        p_portabilidad = extraer_precio(prices_node.get('portIn',         {}).get('relatedPrice', []))
        p_renovacion   = extraer_precio(prices_node.get('renew',          {}).get('relatedPrice', []))
        p_nueva        = extraer_precio(prices_node.get('newConnection',  {}).get('relatedPrice', []))
        cuota          = extraer_precio(prices_node.get('renew',          {}).get('relatedPrice', []), 'installmentPrice')

        lista_wom.append({
            "operadora"          : OPERADORA,
            "producto"           : g_data.get('name'),
            "marca"              : marca,
            # Precio estandarizado para consolidación con otras operadoras
            "precio_lista"       : p_normal,
            "precio_oferta"      : p_portabilidad,   # precio más competitivo (portabilidad)
            # Precios adicionales exclusivos de WOM
            "precio_renovacion"  : p_renovacion,
            "precio_linea_nueva" : p_nueva,
            "cuota_arriendo"     : cuota,
            "sku"                : hijo.get('child_sku'),
            "fecha_extraccion"   : FECHA_HOY,
        })

df_wom = pd.DataFrame(lista_wom)
print(f"✅ Éxito — {len(df_wom)} equipos extraídos de {OPERADORA}.")

Conectando con la API de WOM...
✅ Éxito — 120 equipos extraídos de WOM.


In [3]:
# Limpieza y tipos de datos
## 1. Convertir columnas de precio a numérico
cols_precio = ['precio_lista', 'precio_oferta', 'precio_renovacion', 'precio_linea_nueva', 'cuota_arriendo']
for col in cols_precio:
    df_wom[col] = pd.to_numeric(df_wom[col], errors='coerce').round().astype('Int64')

## 2. Estandarizar marcas
df_wom['marca'] = df_wom['marca'].str.strip().str.title()

## 3. Calcular descuento portabilidad vs precio normal
df_wom['descuento_pct'] = (
    (df_wom['precio_lista'] - df_wom['precio_oferta'])
    / df_wom['precio_lista'] * 100
).round(1)

## 4. Calcular ahorro portabilidad vs renovación (insight exclusivo WOM)
df_wom['ahorro_portabilidad_vs_renovacion'] = df_wom['precio_renovacion'] - df_wom['precio_oferta']

## 5. Eliminar duplicados por SKU
df_wom = df_wom.drop_duplicates(subset='sku').reset_index(drop=True)

print("── Tipos de datos después de limpieza:")
print(df_wom.dtypes)
print(f"\n── Filas finales: {len(df_wom)}")
print(f"── Marcas encontradas: {sorted(df_wom['marca'].dropna().unique())}")

── Tipos de datos después de limpieza:
operadora                             object
producto                              object
marca                                 object
precio_lista                           Int64
precio_oferta                          Int64
precio_renovacion                      Int64
precio_linea_nueva                     Int64
cuota_arriendo                         Int64
sku                                   object
fecha_extraccion                      object
descuento_pct                        Float64
ahorro_portabilidad_vs_renovacion      Int64
dtype: object

── Filas finales: 120
── Marcas encontradas: ['Apple', 'Honor', 'Infinix', 'Motorola', 'Samsung', 'Tecno', 'Xiaomi', 'Zte']


In [4]:
# Vista previa del dataset limpio
display(df_wom.head(10))

,operadora,producto,marca,precio_lista,precio_oferta,precio_renovacion,precio_linea_nueva,cuota_arriendo,sku,fecha_extraccion,descuento_pct,ahorro_portabilidad_vs_renovacion
0,WOM,Redmi Note 12 Pro 5G Black,Xiaomi,228000,204000,459990,204000,16000,001.002.2376,2026-06-30,10.5,255990
1,WOM,Redmi Note 12 Pro 5G Blue,Xiaomi,228000,204000,459990,204000,23000,001.002.2377,2026-06-30,10.5,255990
2,WOM,Redmi Note 12 Pro 5G White,Xiaomi,228000,204000,459990,204000,23000,001.002.2545,2026-06-30,10.5,255990
3,WOM,Apple iPhone Air 5G 512GB Space Black,Apple,1480800,1360800,1559990,1360800,51300,001.002.2885,2026-06-30,8.1,199190
4,WOM,ZTE Blade A56 Pro 4G 128GB Grey,Zte,115200,103200,149990,103200,7600,001.002.2822,2026-06-30,10.4,46790
5,WOM,Samsung Galaxy A56 5G 128GB Black,Samsung,340800,304800,499990,304800,19600,001.002.2793,2026-06-30,10.6,195190
6,WOM,Samsung Galaxy S25 FE 5G 128GB Dark Blue,Samsung,619200,499200,799990,499200,24050,001.002.2848,2026-06-30,19.4,300790
7,WOM,Samsung Galaxy S25 FE 5G 128GB Jet Black,Samsung,619200,499200,799990,499200,24050,001.002.2964,2026-06-30,19.4,300790
8,WOM,Samsung Galaxy A17 5G 128GB Black,Samsung,163200,139200,229990,139200,11500,001.002.2845,2026-06-30,14.7,90790
9,WOM,Samsung Galaxy Z Flip7 5G 256GB Jetblack,Samsung,964800,844800,1399990,844800,36200,001.002.2847,2026-06-30,12.4,555190


In [5]:
# Estadísticas básicas
print("── Resumen por marca:")
resumen = (
    df_wom.groupby('marca')
    .agg(
        cantidad                    = ('producto', 'count'),
        precio_normal_prom          = ('precio_lista',  'mean'),
        precio_portabilidad_prom    = ('precio_oferta', 'mean'),
        ahorro_portabilidad_prom    = ('ahorro_portabilidad_vs_renovacion', 'mean'),
        descuento_pct_prom          = ('descuento_pct', 'mean'),
    )
    .round(0)
    .sort_values('cantidad', ascending=False)
)
display(resumen)


── Resumen por marca:


,cantidad,precio_normal_prom,precio_portabilidad_prom,ahorro_portabilidad_prom,descuento_pct_prom
marca,,,,,
Apple,33,1141964.0,1055055.0,155238.0,8.0
Xiaomi,32,373100.0,322925.0,148002.0,13.0
Motorola,23,240574.0,207704.0,168373.0,11.0
Honor,15,257813.0,228853.0,114470.0,11.0
Samsung,8,490200.0,416700.0,274540.0,14.0
Infinix,4,168000.0,150000.0,124990.0,10.0
Tecno,3,165600.0,145600.0,124390.0,12.0
Zte,2,142800.0,124800.0,115190.0,12.0


In [6]:
## Cambio nombres columnas
df_wom = df_wom.rename(columns={'precio_lista': 'precio_prepago'})
df_wom = df_wom.rename(columns={'precio_renovacion': 'precio_normal'})
df_wom = df_wom.rename(columns={'precio_oferta': 'precio_portabilidad/renovacion'})
df_wom = df_wom.drop(columns=['descuento_pct', 'ahorro_portabilidad_vs_renovacion'])

In [8]:
# Exportar CSV
nombre_archivo = f"wom_{FECHA_HOY}.csv"
# Solo columnas estándar para consolidación con otras operadoras
df_wom.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')
print(f"✅ Archivo guardado con {len(df_wom)} registros.")
print("   Columnas exportadas:", list(df_wom.columns))

✅ Archivo guardado con 120 registros.
   Columnas exportadas: ['operadora', 'producto', 'marca', 'precio_prepago', 'precio_portabilidad/renovacion', 'precio_normal', 'precio_linea_nueva', 'cuota_arriendo', 'sku', 'fecha_extraccion']
